# 01 — Build a Tokenizer from Scratch

**Topic:** Build a tokenizer  
**Audience:** AI PMs, founders, ML engineers, and portfolio builders.  
**How to use this notebook:** run the toy implementation first, then replace the toy data/model with your company data or project data.

## Mental model

This notebook is part of a 32-part LLM systems implementation path. Each notebook follows the same pattern:

1. Product intuition: what capability this unlocks.
2. Core idea: the minimum math or architecture you need.
3. Implementation from scratch or near-scratch.
4. Production notes: cost, risk, reliability, and founder decisions.
5. Portfolio extension: how to turn this into a public project.


In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Tokenization is the first product constraint in an LLM system. It affects:

- **Cost:** APIs and serving systems usually charge or budget by token.
- **Latency:** longer token sequences mean more attention work and larger KV caches.
- **Quality:** poor tokenization hurts rare words, code, names, multilingual text, and domain terms.
- **Data privacy:** tokenization determines what is stored in logs, traces, and embeddings.

You will build a small Byte Pair Encoding style tokenizer. It is not production-grade, but it teaches the core mechanics behind subword tokenizers.


In [ ]:
corpus = (DATA / 'tiny_corpus.txt').read_text().lower().splitlines()
corpus[:3]


## Step 1: whitespace + punctuation tokenizer

This baseline is useful for search, preprocessing, and debug views. It is not enough for modern LMs because unknown words explode.


In [ ]:
def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

for line in corpus[:2]:
    print(line)
    print(basic_tokenize(line))


## Step 2: train a tiny BPE tokenizer

BPE starts with characters and repeatedly merges frequent adjacent pairs. The result is a vocabulary of reusable subword pieces.


In [ ]:
from collections import Counter, defaultdict

class TinyBPE:
    def __init__(self, vocab_size=80, end_token='</w>'):
        self.vocab_size = vocab_size
        self.end_token = end_token
        self.merges = []
        self.token_to_id = {}
        self.id_to_token = {}

    def _word_to_symbols(self, word):
        return tuple(list(word) + [self.end_token])

    def _get_pair_counts(self, word_freq):
        counts = Counter()
        for symbols, freq in word_freq.items():
            for a, b in zip(symbols, symbols[1:]):
                counts[(a, b)] += freq
        return counts

    def _merge_pair(self, pair, word_freq):
        merged = ''.join(pair)
        new_word_freq = {}
        for symbols, freq in word_freq.items():
            out = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == pair:
                    out.append(merged)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            new_word_freq[tuple(out)] = freq
        return new_word_freq

    def fit(self, texts):
        words = []
        for text in texts:
            words.extend(basic_tokenize(text))
        word_freq = Counter(self._word_to_symbols(w) for w in words)
        base_vocab = set(sym for word in word_freq for sym in word)
        while len(base_vocab) + len(self.merges) < self.vocab_size:
            pair_counts = self._get_pair_counts(word_freq)
            if not pair_counts:
                break
            best_pair, count = pair_counts.most_common(1)[0]
            if count < 2:
                break
            self.merges.append(best_pair)
            word_freq = self._merge_pair(best_pair, word_freq)
            base_vocab.add(''.join(best_pair))
        vocab = ['<pad>', '<unk>'] + sorted(base_vocab)
        self.token_to_id = {tok: i for i, tok in enumerate(vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}
        return self

    def encode_word(self, word):
        symbols = list(word.lower()) + [self.end_token]
        for pair in self.merges:
            merged = ''.join(pair)
            out = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == pair:
                    out.append(merged)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            symbols = out
        return symbols

    def encode(self, text):
        pieces = []
        for word in basic_tokenize(text):
            pieces.extend(self.encode_word(word))
        return [self.token_to_id.get(p, self.token_to_id['<unk>']) for p in pieces]

    def decode(self, ids):
        pieces = [self.id_to_token.get(i, '<unk>') for i in ids]
        text = ''.join(pieces).replace(self.end_token, ' ')
        return text.strip()

bpe = TinyBPE(vocab_size=70).fit(corpus)
print('First 15 merges:', bpe.merges[:15])
print('Vocab size:', len(bpe.token_to_id))


In [ ]:
text = 'A tokenizer turns company documents into stable ids.'
ids = bpe.encode(text)
print('text:', text)
print('ids:', ids)
print('decoded:', bpe.decode(ids))


## Production notes

For company use, write tests for:

- Round-trip behavior: `decode(encode(x))` should be acceptable for your use case.
- Domain vocabulary: product names, legal terms, code identifiers, SKU names.
- Multilingual behavior.
- Cost impact: compare characters per token across user segments.
- Privacy: never log raw sensitive text unless your policy allows it.


In [ ]:
def tokenizer_report(texts, tokenizer):
    rows = []
    for t in texts:
        ids = tokenizer.encode(t)
        rows.append({'text': t, 'chars': len(t), 'tokens': len(ids), 'chars_per_token': len(t)/max(1,len(ids))})
    return pd.DataFrame(rows)

report = tokenizer_report([
    'refund policy',
    'multi-head latent attention reduces kv cache size',
    'ABN 12 345 678 901',
    'SuperLongCompanySpecificProductNameV2'
], bpe)
report


## Portfolio / company reuse checklist

To reuse this notebook in a real project:

- Replace the toy corpus/data with a small but representative sample from your domain.
- Add a held-out evaluation set before optimizing anything.
- Track failure cases by category, not just aggregate accuracy/loss.
- Decide whether this is a prototype-only component, a production component, or a learning artifact.
- Document assumptions in a model card or system card.


## References to review after implementation

Use the implementation above first, then read the original papers/docs to connect the code to production systems. The README contains a consolidated reference list for the full pack.
